# Phi-2 Mechanistic Interpretability with TDA

This notebook visualizes the topological structure of phi-2 weight matrices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle
from persim import plot_diagrams
import umap
from sklearn.decomposition import PCA

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print("📊 Loaded visualization libraries")

## 1. Load TDA Results

In [ ]:
# Load TDA results
results_dir = Path("tda_results")

# Load Betti analysis
with open(results_dir / "betti_analysis.json", "r") as f:
    betti_analysis = json.load(f)

# Load persistence diagrams
with open(results_dir / "persistence_diagrams.pkl", "rb") as f:
    ph_results = pickle.load(f)

# Load dimension analysis
with open(Path("point_clouds") / "dimension_analysis.json", "r") as f:
    dim_analysis = json.load(f)

print(f"📁 Loaded results for {len(betti_analysis)} strata")
print(f"📁 Strata: {list(betti_analysis.keys())}")

## 2. Betti Number Analysis

In [ ]:
# Create comprehensive Betti number visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

strata = list(betti_analysis.keys())
beta_0 = [betti_analysis[s]['beta_0'] for s in strata]
beta_1 = [betti_analysis[s]['beta_1'] for s in strata]
max_pers_0 = [betti_analysis[s]['max_persistence_H0'] for s in strata]
max_pers_1 = [betti_analysis[s]['max_persistence_H1'] for s in strata]

# β₀ (Connected Components)
axes[0,0].bar(range(len(strata)), beta_0, color='skyblue', alpha=0.7)
axes[0,0].set_title('β₀: Connected Components by Stratum')
axes[0,0].set_ylabel('Number of Components')
axes[0,0].set_xticks(range(len(strata)))
axes[0,0].set_xticklabels(strata, rotation=45, ha='right')

# β₁ (Loops)
axes[0,1].bar(range(len(strata)), beta_1, color='lightcoral', alpha=0.7)
axes[0,1].set_title('β₁: Loops by Stratum')
axes[0,1].set_ylabel('Number of Loops')
axes[0,1].set_xticks(range(len(strata)))
axes[0,1].set_xticklabels(strata, rotation=45, ha='right')

# Max Persistence H₀
axes[1,0].bar(range(len(strata)), max_pers_0, color='lightgreen', alpha=0.7)
axes[1,0].set_title('Max Persistence H₀ by Stratum')
axes[1,0].set_ylabel('Max Persistence')
axes[1,0].set_xticks(range(len(strata)))
axes[1,0].set_xticklabels(strata, rotation=45, ha='right')

# Max Persistence H₁
axes[1,1].bar(range(len(strata)), max_pers_1, color='orange', alpha=0.7)
axes[1,1].set_title('Max Persistence H₁ by Stratum')
axes[1,1].set_ylabel('Max Persistence')
axes[1,1].set_xticks(range(len(strata)))
axes[1,1].set_xticklabels(strata, rotation=45, ha='right')

plt.tight_layout()
plt.savefig(results_dir / "comprehensive_betti_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"📊 Total β₀ across all strata: {sum(beta_0)}")
print(f"📊 Total β₁ across all strata: {sum(beta_1)}")

## 3. Persistence Diagrams

In [ ]:
# Create a grid of persistence diagrams
n_strata = len(ph_results)
cols = 3
rows = (n_strata + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
axes = axes.flatten() if n_strata > 1 else [axes]

for i, (stratum_name, result) in enumerate(ph_results.items()):
    if i < len(axes):
        diagrams = result['diagrams']
        plot_diagrams(diagrams, show=False, ax=axes[i])
        axes[i].set_title(f'{stratum_name}\n{result["point_count"]} points, {result["dimension"]}D')
        axes[i].grid(True, alpha=0.3)

# Hide empty subplots
for i in range(len(ph_results), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig(results_dir / "all_persistence_diagrams.png", dpi=300, bbox_inches='tight')
plt.show()

## 4. Dimensionality Analysis

In [ ]:
# Analyze intrinsic vs ambient dimensions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

strata = list(dim_analysis.keys())
ambient_dims = [dim_analysis[s]['ambient_dim'] for s in strata]
effective_dims_95 = [dim_analysis[s]['effective_dim_95'] for s in strata]
effective_dims_99 = [dim_analysis[s]['effective_dim_99'] for s in strata]

x = np.arange(len(strata))
width = 0.25

# Dimension comparison
ax1.bar(x - width, ambient_dims, width, label='Ambient Dimension', alpha=0.7)
ax1.bar(x, effective_dims_95, width, label='Effective Dim (95%)', alpha=0.7)
ax1.bar(x + width, effective_dims_99, width, label='Effective Dim (99%)', alpha=0.7)

ax1.set_xlabel('Stratum')
ax1.set_ylabel('Dimension')
ax1.set_title('Intrinsic vs Ambient Dimensions')
ax1.set_xticks(x)
ax1.set_xticklabels(strata, rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dimension reduction ratio
reduction_ratios = [effective_dims_95[i] / ambient_dims[i] for i in range(len(strata))]
ax2.bar(strata, reduction_ratios, color='purple', alpha=0.7)
ax2.set_xlabel('Stratum')
ax2.set_ylabel('Dimension Reduction Ratio')
ax2.set_title('Effective/Ambient Dimension Ratio')
ax2.set_xticklabels(strata, rotation=45, ha='right')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='No Reduction')
ax2.legend()

plt.tight_layout()
plt.savefig(results_dir / "dimensionality_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

print("📊 Dimensionality Summary:")
for i, stratum in enumerate(strata):
    print(f"  {stratum}: {ambient_dims[i]}D → {effective_dims_95[i]}D ({reduction_ratios[i]:.2f} ratio)")

## 5. Point Cloud Visualization (UMAP)

In [ ]:
# Load and visualize point clouds with UMAP
point_clouds_dir = Path("point_clouds")

def visualize_stratum_umap(stratum_name, max_points=5000):
    """Visualize a stratum using UMAP."""
    file_path = point_clouds_dir / f"{stratum_name}.npz"
    if not file_path.exists():
        print(f"⚠️  File not found: {file_path}")
        return
    
    data = np.load(file_path)
    points = data['points']
    
    # Subsample if too many points
    if len(points) > max_points:
        indices = np.random.choice(len(points), max_points, replace=False)
        points = points[indices]
    
    # Apply UMAP
    reducer = umap.UMAP(n_components=2, random_state=42)
    embedding = reducer.fit_transform(points)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.scatter(embedding[:, 0], embedding[:, 1], alpha=0.6, s=1)
    plt.title(f'UMAP Visualization: {stratum_name}\n{len(points)} points')
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(results_dir / f"umap_{stratum_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

# Visualize a few key strata
key_strata = ['Q_stratum', 'K_stratum', 'V_stratum', 'embedding']
for stratum in key_strata:
    if stratum in [f.stem for f in point_clouds_dir.glob("*.npz")]:
        visualize_stratum_umap(stratum)

## 6. Summary Report

In [ ]:
# Generate summary report
print("🔍 PHI-2 MECHANISTIC INTERPRETABILITY ANALYSIS")
print("=" * 50)
print()

print("📊 TOPOLOGY SUMMARY:")
total_beta_0 = sum(betti_analysis[s]['beta_0'] for s in betti_analysis)
total_beta_1 = sum(betti_analysis[s]['beta_1'] for s in betti_analysis)
print(f"  Total Connected Components (β₀): {total_beta_0}")
print(f"  Total Loops (β₁): {total_beta_1}")
print(f"  Analyzed Strata: {len(betti_analysis)}")
print()

print("📏 DIMENSIONALITY SUMMARY:")
for stratum in dim_analysis:
    info = dim_analysis[stratum]
    ratio = info['effective_dim_95'] / info['ambient_dim']
    print(f"  {stratum}: {info['ambient_dim']}D → {info['effective_dim_95']}D ({ratio:.2f})")
print()

print("🎯 KEY FINDINGS:")
# Find most connected stratum
most_connected = max(betti_analysis.items(), key=lambda x: x[1]['beta_0'])
print(f"  Most Connected Stratum: {most_connected[0]} ({most_connected[1]['beta_0']} components)")

# Find stratum with most loops
most_loops = max(betti_analysis.items(), key=lambda x: x[1]['beta_1'])
print(f"  Most Loops: {most_loops[0]} ({most_loops[1]['beta_1']} loops)")

# Find most compressed stratum
most_compressed = min(dim_analysis.items(), key=lambda x: x[1]['effective_dim_95'] / x[1]['ambient_dim'])
compression_ratio = most_compressed[1]['effective_dim_95'] / most_compressed[1]['ambient_dim']
print(f"  Most Compressed Stratum: {most_compressed[0]} ({compression_ratio:.2f} ratio)")

print()
print("✅ Analysis Complete! Check the tda_results/ directory for all visualizations.")